# Exploratory Analysis & Feature Engineering

This notebook takes the SQL-built customer-observation dataset and creates the behavioral features used by the final churn models. The source dataset already contains churn labels, demographics, all eight 28-day behavior counts, and the synthetic subscription features.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_PATH = Path("../data/churn_training_final.csv")
OUTPUT_PATH = Path("../data/churn_modeling_ready.csv")


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["observation_date"])

df["country"] = df["country"].fillna("Unknown")

df["is_churn"] = (
    df["is_churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "t": 1, "true": 1, "1": 1,
        "f": 0, "false": 0, "0": 0
    })
)

print("Shape:", df.shape)
print("Date range:", df["observation_date"].min(), "to", df["observation_date"].max())
print("Unique accounts:", df["account_id"].nunique())
print("Churn observations:", int(df["is_churn"].sum()))
print("Churn rate:", f"{df['is_churn'].mean():.2%}")
display(df.head())


## Data quality checks

The modeling unit is one account-observation date. Duplicate customer-date rows would create leakage or weighting problems, so they are checked explicitly.


In [ ]:
duplicates = df.duplicated(subset=["account_id", "observation_date"]).sum()
print("Duplicate account-date rows:", duplicates)

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_rows"))


## Subscription and churn exploration


In [ ]:
plan_churn = (
    df.groupby("current_plan", observed=True)["is_churn"]
      .agg(observations="count", churners="sum", churn_rate="mean")
      .sort_values("churn_rate", ascending=False)
)

billing_churn = (
    df.groupby("bill_period_months", observed=True)["is_churn"]
      .agg(observations="count", churners="sum", churn_rate="mean")
)

print("Churn by plan")
display(plan_churn)

print("Churn by billing commitment")
display(billing_churn)


## Behavioral feature engineering

The raw event counts are retained. Additional features summarize engagement intensity and normalize negative or response behaviors relative to the customer's overall activity.


In [ ]:
df["total_engagement"] = (
    df["post_per_month"]
    + df["newfriend_per_month"]
    + df["like_per_month"]
    + df["message_per_month"]
    + df["reply_per_month"]
)

df["negative_engagement"] = (
    df["dislike_per_month"]
    + df["unfriend_per_month"]
)

df["total_activity"] = (
    df["post_per_month"]
    + df["newfriend_per_month"]
    + df["like_per_month"]
    + df["adview_per_month"]
    + df["dislike_per_month"]
    + df["unfriend_per_month"]
    + df["message_per_month"]
    + df["reply_per_month"]
)

df["dislike_rate"] = np.where(
    (df["like_per_month"] + df["dislike_per_month"]) > 0,
    df["dislike_per_month"]
    / (df["like_per_month"] + df["dislike_per_month"]),
    0.0
)

df["unfriend_rate"] = np.where(
    (df["newfriend_per_month"] + df["unfriend_per_month"]) > 0,
    df["unfriend_per_month"]
    / (df["newfriend_per_month"] + df["unfriend_per_month"]),
    0.0
)

df["reply_rate"] = np.where(
    df["message_per_month"] > 0,
    df["reply_per_month"] / df["message_per_month"],
    0.0
)

df["negative_activity_share"] = np.where(
    df["total_activity"] > 0,
    df["negative_engagement"] / df["total_activity"],
    0.0
)

df["adview_share"] = np.where(
    df["total_activity"] > 0,
    df["adview_per_month"] / df["total_activity"],
    0.0
)


In [ ]:
engineered_features = [
    "total_engagement",
    "negative_engagement",
    "total_activity",
    "dislike_rate",
    "unfriend_rate",
    "reply_rate",
    "negative_activity_share",
    "adview_share",
]

display(df[engineered_features].describe().T)

comparison = (
    df.groupby("is_churn")[engineered_features + ["account_tenure_days", "current_mrr"]]
      .median()
      .T
)
comparison.columns = ["retained_median", "churned_median"]
display(comparison)


## Final modeling table

Identifiers and the observation date are preserved for splitting and auditing, but they are not used as predictors. The model compares a behavior-only core feature set against an extended feature set that adds subscription economics and plan movement.


In [ ]:
categorical_features = [
    "channel",
    "country",
    "current_plan",
]

numeric_features = [
    "customer_age",
    "account_tenure_days",
    "post_per_month",
    "newfriend_per_month",
    "like_per_month",
    "adview_per_month",
    "dislike_per_month",
    "unfriend_per_month",
    "message_per_month",
    "reply_per_month",
    "total_engagement",
    "negative_engagement",
    "total_activity",
    "dislike_rate",
    "unfriend_rate",
    "reply_rate",
    "negative_activity_share",
    "adview_share",
    "current_mrr",
    "bill_period_months",
    "discount",
    "mrr_change",
    "upgrade_flag",
    "downsell_flag",
    "upgrade_last_90d",
    "downsell_last_90d",
]

model_features = categorical_features + numeric_features

model_df = df[
    ["account_id", "observation_date", "is_churn"] + model_features
].copy()

print("Candidate predictors:", len(model_features))
print("Final modeling shape:", model_df.shape)
print("Missing values:", int(model_df.isna().sum().sum()))
display(model_df.head())


In [ ]:
model_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print("Shape:", model_df.shape)
